In [13]:
from torchvision import transforms, datasets
from torch.utils.data import DataLoader
import torch.nn.functional as F

In [14]:
transforms = transforms.Compose(
    [
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
    ]
)

In [15]:
train_data = datasets.FashionMNIST(root="./.data", train=True, transform=transforms, download=True)
test_data = datasets.FashionMNIST(root="./.data", train=False, transform=transforms, download=True)

In [16]:
train_loader = DataLoader(train_data, batch_size=128, shuffle=True)
test_loader = DataLoader(test_data, batch_size=128, shuffle=False)

In [17]:
import torch
import torch.nn as nn


def nin_block(
    out_channels: int, kernel_size: int, strides: int, padding: int
) -> nn.Sequential:
    return nn.Sequential(
        nn.LazyConv2d(
            out_channels=out_channels,
            kernel_size=kernel_size,
            stride=strides,
            padding=padding,
        ),
        nn.ReLU(),
        nn.Conv2d(in_channels=out_channels, out_channels=out_channels, kernel_size=1),
        nn.ReLU(),
        nn.Conv2d(in_channels=out_channels, out_channels=out_channels, kernel_size=1),
        nn.ReLU(),
    )

In [18]:
class NiN(torch.nn.Module):
    def __init__(self, num_classess=10):
        super().__init__()
        self.net = nn.Sequential(
            nin_block(96, kernel_size=11, strides=4, padding=0),
            nn.MaxPool2d(kernel_size=3, stride=2),
            nin_block(256, kernel_size=5, strides=1, padding=2),
            nn.MaxPool2d(kernel_size=3, stride=2),
            nin_block(384, kernel_size=3, strides=1, padding=1),
            nn.MaxPool2d(kernel_size=3, stride=2),
            nn.Dropout(0.5),
            nin_block(num_classess, kernel_size=3, strides=1, padding=1),
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
        )

    def forward(self, x):
        return self.net(x)

In [19]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using {device} device")

model = NiN().to(device)

Using cuda device


In [20]:
def init_weights(m):
    if type(m) == nn.Linear or type(m) == nn.Conv2d:
        nn.init.xavier_uniform_(m.weight)
model.apply(init_weights)

NiN(
  (net): Sequential(
    (0): Sequential(
      (0): LazyConv2d(0, 96, kernel_size=(11, 11), stride=(4, 4))
      (1): ReLU()
      (2): Conv2d(96, 96, kernel_size=(1, 1), stride=(1, 1))
      (3): ReLU()
      (4): Conv2d(96, 96, kernel_size=(1, 1), stride=(1, 1))
      (5): ReLU()
    )
    (1): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
    (2): Sequential(
      (0): LazyConv2d(0, 256, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
      (1): ReLU()
      (2): Conv2d(256, 256, kernel_size=(1, 1), stride=(1, 1))
      (3): ReLU()
      (4): Conv2d(256, 256, kernel_size=(1, 1), stride=(1, 1))
      (5): ReLU()
    )
    (3): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Sequential(
      (0): LazyConv2d(0, 384, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): ReLU()
      (2): Conv2d(384, 384, kernel_size=(1, 1), stride=(1, 1))
      (3): ReLU()
      (4): Conv2d(384, 384, kernel_size=(1, 1), 

In [21]:
optimizer = torch.optim.SGD(model.parameters(), lr=0.05)

In [22]:
EPOCHS = 10
for epoch in range(EPOCHS):
    model.train()
    total_loss, correct, total = 0, 0, 0
    
    for X, y in train_loader:
        X, y = X.to(device), y.to(device)
        y_hat = model(X)
        loss = F.cross_entropy(y_hat, y)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item() * y.size(0)
        correct += (y_hat.argmax(1) == y).sum().item()
        total += y.size(0)
    
    model.eval()
    test_correct = 0
    with torch.no_grad():
        for X, y in test_loader:
            X, y = X.to(device), y.to(device)
            test_correct += (model(X).argmax(1) == y).sum().item()
    
    train_acc = correct / total
    test_acc = test_correct / len(test_data)
    print(f"Epoch {epoch+1:2d} | "
          f"Loss: {total_loss/total:.4f} | "
          f"Train: {train_acc*100:.1f}% | "
          f"Test: {test_acc*100:.1f}%")

Epoch  1 | Loss: 2.2603 | Train: 16.5% | Test: 15.5%
Epoch  2 | Loss: 1.6699 | Train: 40.9% | Test: 59.7%
Epoch  3 | Loss: 0.9095 | Train: 67.0% | Test: 73.5%
Epoch  4 | Loss: 0.7092 | Train: 73.7% | Test: 78.0%
Epoch  5 | Loss: 0.6114 | Train: 77.3% | Test: 78.3%
Epoch  6 | Loss: 0.5465 | Train: 80.0% | Test: 79.5%
Epoch  7 | Loss: 0.5041 | Train: 81.4% | Test: 83.1%
Epoch  8 | Loss: 0.4659 | Train: 82.9% | Test: 80.5%
Epoch  9 | Loss: 0.4409 | Train: 83.7% | Test: 82.7%
Epoch 10 | Loss: 0.4219 | Train: 84.4% | Test: 84.1%
